# FHOPS Operations Simulation

FHOPS is a forest harvesting operations simulator as well as an optimiser. Simulation is the operational core used to evaluate optimisation results: playback replays explicit machine assignments and reports their consequences; it does **not** optimise or repair assignments.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
examples_dir = cwd if (cwd / 'tiny7').is_dir() else cwd / 'examples'
if not examples_dir.is_dir():
    raise FileNotFoundError('Launch from the FHOPS repository root or examples/.')
repo_root = examples_dir.parent
sys.path.insert(0, str(repo_root / 'src'))

from fhops.evaluation import PlaybackConfig, run_playback
from fhops.scenario.contract import Problem
from fhops.scenario.io import load_scenario
from fhops.scheduling.systems import HarvestSystem, SystemJob, default_system_registry

scenario = load_scenario(examples_dir / 'tiny7' / 'scenario.yaml')
problem = Problem.from_scenario(scenario)
print(f'Loaded {scenario.name}: {len(scenario.blocks)} blocks, {len(scenario.machines)} machines')

Loaded FHOPS Tiny7: 2 blocks, 9 machines


## Block modelling

FHOPS resolves a block as a strategic/tactical schedule abstraction: it carries required work, operating window, landing, terrain and stand/productivity drivers where supplied, and a harvest-system choice. It is not a stem-by-stem model or a spatial traffic simulator.

In [2]:
columns = ['id', 'landing_id', 'work_required', 'earliest_start', 'latest_finish', 'harvest_system_id', 'avg_stem_size_m3', 'volume_per_ha_m3', 'stem_density_per_ha', 'ground_slope_percent']
blocks = pd.DataFrame([block.model_dump() for block in scenario.blocks])
blocks[[column for column in columns if column in blocks.columns]]

,id,landing_id,work_required,earliest_start,latest_finish,harvest_system_id,avg_stem_size_m3,volume_per_ha_m3,stem_density_per_ha,ground_slope_percent
0,B01,L1,2007.912755,1,7,ground_fb_skid,0.426252,269.025614,631.142040,6.48
1,B02,L2,2406.789997,1,7,ground_fb_skid,0.376202,320.151552,851.008994,9.22


## Machine abstraction

Machine roles are capabilities used by system jobs. This lets feasible machine instances be assigned to each job role while productivity and cost rules are applied consistently.

In [3]:
machines = pd.DataFrame([machine.model_dump() for machine in scenario.machines])
display(machines[['id', 'role', 'crew', 'daily_hours', 'operating_cost']])
machines.groupby('role', dropna=False).agg(machines=('id', 'count'), daily_hours=('daily_hours', 'sum'), operating_cost=('operating_cost', 'sum'))

,id,role,crew,daily_hours,operating_cost
0,H1,feller_buncher,CH1,24.0,950.0
1,H2,feller_buncher,CH2,24.0,1050.0
2,H3,grapple_skidder,CH3,24.0,1045.0
3,H4,processor,CH4,24.0,902.5
4,H5,processor,CH5,24.0,997.5
5,H6,processor,CH6,24.0,1092.5
6,H7,loader,CH7,24.0,855.0
7,H8,loader,CH8,24.0,945.0
8,H9,loader,CH9,24.0,1035.0


,machines,daily_hours,operating_cost
role,,,
feller_buncher,2,48.0,2000.0
grapple_skidder,1,24.0,1045.0
loader,3,72.0,2835.0
processor,3,72.0,2992.5


## Harvest system workflow

Systems impose workflow order and prerequisites. Role counts, role headstarts, and loader batch volume express staged work and buffering constraints used by playback.

In [4]:
system_id = scenario.blocks[0].harvest_system_id
system_registry = scenario.harvest_systems or default_system_registry()
selected_system = system_registry[system_id]
display(pd.DataFrame([{'name': job.name, 'machine_role': job.machine_role, 'prerequisites': ', '.join(job.prerequisites) or '(start)'} for job in selected_system.jobs]))
pd.DataFrame([{'system_id': selected_system.system_id, 'role_counts': dict(selected_system.role_counts or {}), 'role_headstart_shifts': dict(selected_system.role_headstart_shifts or {}), 'loader_batch_volume_m3': selected_system.loader_batch_volume_m3}])

,name,machine_role,prerequisites
0,felling,feller_buncher,(start)
1,primary_transport,grapple_skidder,felling
2,processing,processor,primary_transport
3,loading,loader,processing


,system_id,role_counts,role_headstart_shifts,loader_batch_volume_m3
0,ground_fb_skid,"{'feller_buncher': 2, 'grapple_skidder': 1, 'r...","{'roadside_processor': 0.0, 'loader': 0.0}",30.0


## Built-in registry and a local extension

The registry below is observed directly from FHOPS. Its presets reflect documented common operating contexts and cited study configurations; this is not a claim of exhaustive industry coverage. The custom profile is an isolated construction suitable for a scenario's `harvest_systems` map, without mutating Tiny7 or solving.

In [5]:
registry = default_system_registry()
registry_table = pd.DataFrame([{'id': system.system_id, 'environment': system.environment, 'role_chain': ' -> '.join(job.machine_role for job in system.jobs)} for system in registry.values()])
display(registry_table.sort_values('id'))
observed_environments = sorted(registry_table['environment'].dropna().unique())
observed_categories = [category for category in ('ground-based', 'CTL', 'steep', 'cable/skyline', 'helicopter') if any(category.lower() in environment.lower() for environment in observed_environments)]
print('Observed registry categories:', observed_categories)
print('Observed environment values:', observed_environments)
custom_system = HarvestSystem(system_id='local_ground_trial', environment='local documented trial context', jobs=(SystemJob('felling', 'feller_buncher', ()), SystemJob('transport', 'grapple_skidder', ('felling',))))
extended_registry = {**scenario.harvest_systems, custom_system.system_id: custom_system}
print('Tiny7 unchanged:', custom_system.system_id not in scenario.harvest_systems)
print('Extension available locally:', extended_registry[custom_system.system_id].system_id)

,id,environment,role_chain
14,cable_highlead_tn147,cable-highlead skyline,hand_faller -> grapple_yarder -> processor -> ...
28,cable_micro_christie,cable-short-span skyline,hand_faller -> skyline_christie_tn173 -> proce...
26,cable_micro_ecologger,cable-short-span skyline,hand_faller -> skyline_ecologger_tn173 -> proc...
27,cable_micro_gabriel,cable-short-span skyline,hand_faller -> skyline_gabriel_tn173 -> proces...
31,cable_micro_hi_skid,cable-short-span skyline,hand_faller -> skyline_hi_skid
29,cable_micro_teletransporteur,cable-short-span skyline,hand_faller -> skyline_teletransporteur_tn173 ...
30,cable_micro_timbermaster,cable-short-span skyline,hand_faller -> skyline_timbermaster_tn173 -> p...
18,cable_partial_tr127_block1,cable-standing skyline,hand_or_mech_faller -> skyline_yarder -> proce...
19,cable_partial_tr127_block5,cable-standing skyline,hand_or_mech_faller -> skyline_yarder -> proce...
12,cable_running,cable-running skyline,hand_or_mech_faller -> grapple_yarder -> proce...


Observed registry categories: ['ground-based', 'steep', 'helicopter']
Observed environment values: ['cable-highlead skyline', 'cable-running salvage', 'cable-running skyline', 'cable-short-span skyline', 'cable-standing skyline', 'commercial thinning', 'cut-to-length', 'ground-based', 'ground-based salvage', 'helicopter', 'steep-slope mechanised']
Tiny7 unchanged: True
Extension available locally: local_ground_trial


## Productivity functions

Tiny7's dataset README identifies Lahrsen, ADV6N7 grapple-skidder, Berry 2019 processor, and TN-261 loader models. Public function docstrings identify each regression's source and inputs. These source-based regressions should be used within their stated input envelopes; they are not universal truth.

In [6]:
import fhops.productivity as productivity
exports = pd.DataFrame({'export': productivity.__all__})
exports['module'] = exports['export'].map(lambda name: getattr(getattr(productivity, name), '__module__', '(type or constant)'))
print(f'Computed public productivity exports: {len(exports)}')
display(exports.groupby('module', dropna=False).size().rename('exports').reset_index())
display(exports)

Computed public productivity exports: 170


,module,exports
0,(type or constant),7
1,fhops.productivity.cable_logging,35
2,fhops.productivity.eriksson2014,2
3,fhops.productivity.forwarder_bc,3
4,fhops.productivity.ghaffariyan2019,4
5,fhops.productivity.grapple_bc,25
6,fhops.productivity.harvester_ctl,6
7,fhops.productivity.kellogg_bettinger1994,2
8,fhops.productivity.lahrsen2025,5
9,fhops.productivity.laitila2020,1


,export,module
0,Fncy12ProductivityVariant,fhops.productivity.cable_logging
1,LahrsenModel,fhops.productivity.lahrsen2025
2,ProductivityEstimate,fhops.productivity.lahrsen2025
3,ProductivityDistributionEstimate,fhops.productivity.lahrsen2025
4,estimate_productivity,fhops.productivity.lahrsen2025
...,...,...
165,get_labelle_huss_automatic_bucking_adjustment,fhops.productivity.processor_loader
166,BerryLogGradeStat,fhops.productivity.processor_loader
167,get_berry_log_grade_stats,fhops.productivity.processor_loader
168,get_berry_log_grade_metadata,fhops.productivity.processor_loader


In [7]:
from dataclasses import asdict
from fhops.productivity import ADV6N7DeckingMode, estimate_grapple_skidder_productivity_adv6n7, estimate_loader_forwarder_productivity_tn261, estimate_processor_productivity_berry2019, estimate_productivity

block = scenario.blocks[0]
results = {'Lahrsen feller-buncher': estimate_productivity(avg_stem_size=block.avg_stem_size_m3, volume_per_ha=block.volume_per_ha_m3, stem_density=block.stem_density_per_ha, ground_slope=block.ground_slope_percent), 'ADV6N7 grapple skidder': estimate_grapple_skidder_productivity_adv6n7(skidding_distance_m=85.0, decking_mode=ADV6N7DeckingMode.SKIDDER_LOADER), 'Berry 2019 processor': estimate_processor_productivity_berry2019(piece_size_m3=block.avg_stem_size_m3), 'TN-261 loader': estimate_loader_forwarder_productivity_tn261(piece_size_m3=1.05, external_distance_m=115.0, slope_percent=8.0, bunched=True, delay_multiplier=0.95)}
pd.DataFrame([{'model': name, **asdict(result)} for name, result in results.items()])

,model,avg_stem_size,volume_per_ha,stem_density,ground_slope,predicted_m3_per_pmh,ranges,out_of_range,decking_mode,skidding_distance_m,...,delay_multiplier,delay_free_productivity_m3_per_pmh,piece_size_m3,tree_form_category,carrier_profile,external_distance_m,slope_percent,bunched,slope_multiplier,bunched_multiplier
0,daily,0.426252,269.025614,631.14204,6.48,49.781191,"{'avg_stem_size': {'min': 0.09, 'max': 1.32, '...",(),NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ADV6N7 grapple skidder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skidder_loader,85.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Berry 2019 processor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.91,26.090944,0.426252,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,TN-261 loader,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.95,56.378984,1.050000,NaN,NaN,115.0,8.0,True,0.976,1.0


## Deterministic simulation playback

This is a clearly labelled full-work example for one block. It stages the same required work through felling, primary transport, processing, and loading on successive days, exposing records, day summaries, delivered work, staged role inventory, remaining work, and mobilisation signals.

In [8]:
playback_block = scenario.blocks[0]
full_work = playback_block.work_required
staged_assignments = pd.DataFrame([{'machine_id': 'H1', 'block_id': playback_block.id, 'day': 1, 'shift_id': 'S1', 'assigned': 1, 'production': full_work}, {'machine_id': 'H3', 'block_id': playback_block.id, 'day': 2, 'shift_id': 'S1', 'assigned': 1, 'production': full_work}, {'machine_id': 'H4', 'block_id': playback_block.id, 'day': 3, 'shift_id': 'S1', 'assigned': 1, 'production': full_work}, {'machine_id': 'H7', 'block_id': playback_block.id, 'day': 4, 'shift_id': 'S1', 'assigned': 1, 'production': full_work}])
playback = run_playback(problem, staged_assignments, config=PlaybackConfig())
records = pd.DataFrame([asdict(record) for record in playback.records])
display(records[['day', 'shift_id', 'machine_id', 'machine_role', 'block_id', 'production_units', 'mobilisation_cost', 'metadata']])
display(pd.DataFrame([asdict(summary) for summary in playback.day_summaries]))
print('Delivered work:', playback.delivered_total)
print('Remaining work:', playback.remaining_work_total)
print('Sequencing and staged inventory:', playback.sequencing_debug)

,day,shift_id,machine_id,machine_role,block_id,production_units,mobilisation_cost,metadata
0,1,S1,H1,feller_buncher,B01,2007.912755,None,"{'hours_source': 'machine_daily_hours', 'produ..."
1,2,S1,H3,grapple_skidder,B01,2007.912755,None,"{'hours_source': 'machine_daily_hours', 'produ..."
2,3,S1,H4,processor,B01,2007.912755,None,"{'hours_source': 'machine_daily_hours', 'produ..."
3,4,S1,H7,loader,B01,2007.912755,None,"{'hours_source': 'machine_daily_hours', 'produ..."


,day,available_hours,total_hours,production_units,mobilisation_cost,completed_blocks,idle_hours,blackout_conflicts,sequencing_violations,utilisation_ratio,sample_id,downtime_hours,downtime_events,weather_severity_total
0,1,216.0,24.0,2007.912755,0.0,0,192.0,0,0,0.111111,0,0.0,0,0.0
1,2,216.0,24.0,2007.912755,0.0,0,192.0,0,0,0.111111,0,0.0,0,0.0
2,3,216.0,24.0,2007.912755,0.0,0,192.0,0,0,0.111111,0,0.0,0,0.0
3,4,216.0,24.0,2007.912755,0.0,1,192.0,0,0,0.111111,0,0.0,0,0.0
4,5,216.0,0.0,0.000000,0.0,0,216.0,0,0,0.000000,0,0.0,0,0.0
5,6,216.0,0.0,0.000000,0.0,0,216.0,0,0,0.000000,0,0.0,0,0.0
6,7,216.0,0.0,0.000000,0.0,0,216.0,0,0,0.000000,0,0.0,0,0.0


Delivered work: 2007.912755
Remaining work: 2406.789997
Sequencing and staged inventory: {'sequencing_violation_count': 0, 'role_inventory_totals': {'feller_buncher': 0.0, 'grapple_skidder': 0.0, 'loader': 2007.912755, 'processor': 0.0}, 'role_remaining_totals': {'feller_buncher': 2406.789997, 'grapple_skidder': 2406.789997, 'loader': 2406.789997, 'processor': 2406.789997}, 'completed_blocks': 1, 'remaining_work_total': 2406.789997, 'delivered_total': 2007.912755, 'sequencing_status': 'clean'}


## Sequencing and buffering validation

Here loading is deliberately scheduled before processing. Playback records the actual violation and prevents unbuffered production. The observed inventory diagnostic describes the model's staging semantics; it does not promise same-day material transfer.

In [9]:
loader_first = pd.DataFrame([{'machine_id': 'H7', 'block_id': playback_block.id, 'day': 1, 'shift_id': 'S1', 'assigned': 1, 'production': 30.0}])
invalid_playback = run_playback(problem, loader_first, config=PlaybackConfig())
invalid_records = pd.DataFrame([asdict(record) for record in invalid_playback.records])
display(invalid_records[['machine_id', 'machine_role', 'production_units', 'metadata']])
print('Recorded sequencing debug signal:', invalid_playback.sequencing_debug)
assert invalid_playback.sequencing_debug['sequencing_first_violation_reason'] == 'inventory'
assert invalid_playback.sequencing_debug['sequencing_violation_count'] > 0

,machine_id,machine_role,production_units,metadata
0,H7,loader,0.0,"{'hours_source': 'machine_daily_hours', 'produ..."


Recorded sequencing debug signal: {'sequencing_violation_count': 1, 'sequencing_violation_breakdown': {'missing_prereq': 1}, 'sequencing_first_violation_role': 'loader', 'sequencing_first_violation_reason': 'inventory', 'sequencing_first_violation_block': 'B01', 'sequencing_first_violation_day': 1, 'sequencing_first_violation_available_volume': 0.0, 'sequencing_first_violation_required_volume': 30.0, 'role_inventory_totals': {'loader': 0.0, 'processor': 0.0}, 'role_remaining_totals': {'feller_buncher': 4414.702752, 'grapple_skidder': 4414.702752, 'loader': 4414.702752, 'processor': 4414.702752}, 'completed_blocks': 0, 'remaining_work_total': 4414.702752, 'delivered_total': 0.0}


## Applicability and extension

The current registry and productivity helpers span a broad set of supported roles and operating contexts, but remain bounded by their source data and modelling assumptions. For a new context: document the operating system and work sequence; encode block attributes, machine profiles, and production rates or overrides; add or override a `HarvestSystem` in scenario configuration; then replay known assignments before using optimisation. Adding a new regression or coded productivity helper is a different extension: it requires implementing and validating a new public model with its source data, applicability, and tests.

Continue with `02_fhops_solve_compare.ipynb` for optimisation, `03_fhops_playback_kpis.ipynb` for KPI reporting, and `04_fhops_stochastic_what_if.ipynb` for stochastic what-if analysis.